In [ ]:
import os
import re
from functools import reduce
from pathlib import Path


import eda_utils as u
import pandas as pd
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window

# use spark session
spark = (
    SparkSession.builder.master("local[*]")
    .appName("GenPM-eda")
    .config("spark.executor.memory", "48g")
    .config("spark.driver.memory", "32g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.default.parallelism", "8")
    .config("spark.log.level", "ERROR")
    .getOrCreate()
)

user = os.getenv('USER')
eda_data_path = Path(f"/home/{user}/app/apps/generator/data/shared_dir/eda_data")
raw_pm_path = eda_data_path / "raw_pm_data"
pm_kpi_pivot = eda_data_path / "pm_data_pivot"
sample_path = eda_data_path / "sample"
pm_stats_path = eda_data_path / "stats"
pm_agg_path = eda_data_path / "agg"
pm_metadata = eda_data_path / "pm_metadata"

# Consts
MIN_KPI_COVERAGE = 0.9

PRESENTATION = False

# EDA - combined
#### Authors: Grzegorz Wiącek, Paola Jaroń, Mikołaj Lepsy, Filip Antoniak, Maciek Skorupski


## 1. Available data

In [ ]:
def read_and_quickshow(data_path: str, data_name: str) -> DataFrame:
    df: DataFrame = spark.read.parquet(data_path)
    print(f"\t{data_name}")
    print()
    print(f"{df.count()=}")
    df.printSchema()
    df.show(30, truncate=False)
    print()
    return df


pm_df = read_and_quickshow(str(raw_pm_path), "PM Data")
pm_kpi_pivot_df = read_and_quickshow(str(pm_kpi_pivot), "PM Data (pivoted on KPI)")

simple_reports_df = read_and_quickshow(
    str(pm_metadata / "simple_reports"), "Simple Reports - Metadata"
)
kpi_defs_df = read_and_quickshow(
    str(pm_metadata / "kpis_definitions"), "KPI definitions - Metadata"
)

## 2. PM Data
### 2.1 PM Data Preprocessing

PM Data (Long)

In [ ]:
# Data has been deduplicated in raw_data_transform layer

print(f"PM df before preprocessing {pm_df.count()=}")
pm_df = pm_df.dropDuplicates(["bts_id", "distname", "kpi_id", "start_time", "kpi_value"])


print("Null counts in columns")
pm_df.select([f.count(f.when(f.col(c).isNull(), c)).alias(c) for c in pm_df.columns]).show()


# drop null BTSs and Cells
pm_df = pm_df.where((f.col("bts_id").isNotNull()) & (f.col("distname").isNotNull()))

In [ ]:
pm_df_const_kpis = (
    pm_df.groupBy("kpi_id")
    .agg(f.count_distinct(f.col("kpi_value")).alias("distinct_kpi_values"))
    .where(f.col("distinct_kpi_values") == 1)
)
pm_df_const_kpis.select("kpi_id").show(truncate=False)
print(f"{pm_df_const_kpis.count()=}")
# For now, we won't exclude constant kpis (for later modelling, this should be included in dataset creation)


print(f"PM df after preprocessing {pm_df.count()=}")

In [ ]:
print("Distname count per bts_id")
pm_df.groupBy("bts_id").agg(f.count_distinct("distname").alias("distname_dcount")).show(50)

Pm data wide

In [ ]:
# Pm pivoted data processing

grouping_cols = ["bts_id", "distname", "start_time"]

kpi_cols = [c for c in pm_kpi_pivot_df.columns if c not in grouping_cols]

pm_kpi_pivot_df.show(truncate=False)

# Rest of preprocessing

### 2.2 KPI analysis

In [ ]:
kpi_catalog = (
    pm_df.groupBy("kpi_id")
    .agg(
        f.count("*").alias("value_count"),
        f.countDistinct("bts_id").alias("bts_count"),
        f.countDistinct("start_time").alias("timestamp_count"),
        f.min("start_time"),
        f.max("start_time"),
        f.round(f.avg("kpi_value"), 4).alias("kpi_vg"),
        f.round(f.expr("percentile_approx(kpi_value, 0.5)"), 4).alias("val_median"),
        f.round(f.stddev("kpi_value"), 4).alias("kpi_std"),
        f.skewness("kpi_value").alias("skewness"),
        # Coefficient of Variation: high = volatile counter, low = stable gauge/ratio
        (f.stddev("kpi_value") / f.mean("kpi_value")).alias("coeff_variation"),
        f.round(f.min("kpi_value"), 4).alias("kpi_min"),
        f.round(f.max("kpi_value"), 4).alias("kpi_max"),
        f.round(f.sum(f.col("kpi_value").isNull().cast("int")) / f.count("*") * 100, 2).alias(
            "null_pct"
        ),
    )
    .orderBy(f.desc("value_count"))
).cache()

if os.path.exists("/home/{/app/apps/apps/generator/notebooks/eda/data/kpi_catalog"):
    kpi_catalog = spark.read.parquet(
        f"/home/{user}/app/apps/apps/generator/notebooks/eda/data/kpi_catalog"
    )
else:
    kpi_catalog.write.parquet(
        f"/home/{user}/app/apps/apps/generator/notebooks/eda/data/kpi_catalog", mode="overwrite"
    )

kpi_catalog_pdf = kpi_catalog.toPandas()

kpi_catalog.show(1000, truncate=False)

kpi_list = kpi_catalog_pdf["kpi_id"].tolist()

In [ ]:
# Ile jest kpi, których wartość się nie zmienia?
kpi_not_stable_df = kpi_catalog_pdf[kpi_catalog_pdf["kpi_min"] != kpi_catalog_pdf["kpi_max"]]
stable_kpis = kpi_catalog_pdf[kpi_catalog_pdf["kpi_min"] == kpi_catalog_pdf["kpi_max"]]

stable = len(stable_kpis)
print(f"Ilość kpi, których wartość jest stała: {stable}")

display(stable_kpis[stable_kpis["kpi_min"] != 0])

In [ ]:
# What are the kpis per bts?
kpi_per_bts = u.visualize_kpi_bts_coverage(pm_df)

In [ ]:
# Kpi coverage - Time series regularity?
natural_freq_df = u.profiling_regularity(pm_df).cache()
natural_freq_df.groupBy("natural_freq_minutes").count().orderBy("natural_freq_minutes").show(400)

It looks like we have long tail of probably event driven KPIs
- QUESTION TO NOKIANS: Should we expect constant frequency of measurments? Are there _trigger kpis_

In [ ]:
partition_cols = ["kpi_id", "bts_id", "distname"]


pm_with_freq = pm_df.join(natural_freq_df, on=partition_cols, how="left")

output_schema = """
    kpi_id string, bts_id string, distname string, 
    start_time timestamp, kpi_value double, 
    start_hour int, start_date date,
    natural_freq_minutes double
"""


def resample_with_known_freq(pdf: pd.DataFrame) -> pd.DataFrame:
    if pdf.empty:
        return pdf

    # 1. Grab the PRE-CALCULATED frequency
    freq_mins = pdf["natural_freq_minutes"].iloc[0]

    # Check for event-driven KPIs (e.g., skip if frequency > 24 hours or if it's NaN)
    if pd.isna(freq_mins) or freq_mins > 1440:
        return pdf.sort_values("start_time")

    # 2. Grab the static partition values to forward-fill later
    static_kpi = pdf["kpi_id"].iloc[0]
    static_bts = pdf["bts_id"].iloc[0]
    static_dist = pdf["distname"].iloc[0]
    # static_agg = pdf['aggregation_method'].iloc[0] if 'aggregation_method' in pdf.columns else None

    # 3. Sort and set index
    pdf = pdf.sort_values("start_time").set_index("start_time")

    # 4. Resample using the exact known frequency
    freq_string = f"{int(freq_mins)}min"
    pdf_resampled = pdf.resample(freq_string).asfreq()

    # 5. Re-populate the static columns for the new NaN rows
    pdf_resampled["kpi_id"] = static_kpi
    pdf_resampled["bts_id"] = static_bts
    pdf_resampled["distname"] = static_dist
    # pdf_resampled['aggregation_method'] = static_agg
    pdf_resampled["natural_freq_minutes"] = freq_mins

    # 6. Reset index and recalculate time derivatives
    pdf_resampled = pdf_resampled.reset_index()
    pdf_resampled["start_hour"] = pdf_resampled["start_time"].dt.hour
    pdf_resampled["start_date"] = pdf_resampled["start_time"].dt.date

    return pdf_resampled


if os.path.exists(f"/home/{user}/app/apps/apps/generator/notebooks/eda/data/regularized_df"):
    regularized_df = spark.read.parquet(
        f"/home/{user}/app/apps/apps/generator/notebooks/eda/data/regularized_df"
    )
else:
    regularized_df = pm_with_freq.groupBy(*partition_cols).applyInPandas(
        resample_with_known_freq, schema=output_schema
    )
    regularized_df.write.parquet(
        f"/home/{user}/app/apps/apps/generator/notebooks/eda/data/regularized_df",
        mode="overwrite",
    )

regularized_df = regularized_df.cache()

In [ ]:
pm_df_count = pm_df.count()
regularized_df_count = regularized_df.count()
print(f"We got new {pm_df_count - regularized_df_count} filled with NULLs")

print(
    "Proper Imputing is recommended before modelling (for approaches that cannot handle NaN values"
)

### KPI coverage (and cross KPI coverage)

In [ ]:
# KPI coverage
kpi_coverage_df = u.compute_kpi_coverage(pm_df)
kpi_coverage_df.orderBy(f.desc("missing_points")).show()

In [ ]:
# Cross kpi coverage
u.visualize_kpi_overlap(pm_df)

### KPI Grouping based on definitions

In [ ]:
pm_df_with_defs = kpi_catalog.join(kpi_defs_df, on="kpi_id", how="left")
pm_df_with_defs.groupBy(f.col("unit")).count().show()

In [ ]:
mean_like_units = ["%", "bit/s", "kbit/s", "Mbit/s"]
volume_units = ["#"]

# RATIOS: telecom KPI acronyms / categories that are usually percentages
ratio_keywords = [
    "cssr",  # Call Setup Success Rate
    "hosr",  # Handover Success Rate
    "asr",  # Answer Seizure Ratio
    "ccr",  # Call Completion Ratio / related completion KPIs
    "dcr",  # Drop Call Ratio
    "bler",  # Block Error Rate
    "fer",  # Frame Error Rate
    "per",  # Packet Error Rate
    "availability",
    "accessibility",
    "retainability",
    "integrity",
    "utilization",
    "Average Time",
    "Average Duration",
]

# MEAN-LIKE: speed / quality / radio level / delay measurements
mean_like_keywords = [
    "throughput",
    "latency",
    "jitter",
    "rtt",
    "rssi",
    "rsrp",
    "rsrq",
    "sinr",
    "snr",
    "mos",
]

# VOLUME: additive traffic / count style telecom nouns
volume_keywords = [
    "erlang",
    "mou",
    "bytes",
    "octets",
    "attempts",
    "packets",
    "Total Time",
    "Total Duration",
    "volume",
]


def make_pattern(words):
    return r"(?i)(^|[^a-z0-9])(" + "|".join(re.escape(w) for w in words) + r")([^a-z0-9]|$)"


# ratio_pattern   = "(?i)(" + "|".join(ratio_keywords) + ")"

ratio_pattern = make_pattern(ratio_keywords)
mean_like_pattern = make_pattern(mean_like_keywords)
volume_pattern = make_pattern(volume_keywords)

In [ ]:
kpi_classified = (
    pm_df_with_defs.withColumn(
        "unit_match",
        f.when(f.col("unit").isin(*mean_like_units, "mean_like"), "mean_like")
        .when(f.col("unit").isin(*volume_units, "volume"), "volume")
        .otherwise("unknown"),
    )
    .withColumn(
        "keyword_match",
        f.when(f.col("kpi_name").rlike(ratio_pattern), "ratio")
        .when(f.col("kpi_name").rlike(mean_like_pattern), "mean_like")
        .when(f.col("kpi_name").rlike(volume_pattern), "volume")
        .otherwise("unknown"),
    )
    .withColumn(
        "kpi_character",
        # Check unit_match first. If it's not "unknown", use it.
        f.when(f.col("unit_match") != "unknown", f.col("unit_match"))
        # check keywords
        .when(f.col("keyword_match") != "unknown", f.col("keyword_match"))
        # value range fallback
        .when(
            f.col("kpi_min") > 0,
            "mean_like",  # when minimal value is smaller than 0 => 'mean_like' category
        )
        .when(
            (f.col("kpi_min") >= 0) & (f.col("kpi_max") <= 100),
            "ratio",  # when minimal and maximal values are between 0 and 100 => 'ratio' category
        )
        .otherwise("unknown"),
    )
    .withColumn(
        "classification_source",
        f.when(f.col("unit_match").isin("mean_like", "volume"), "unit")
        .when(f.col("keyword_match") != "unknown", "keyword")
        .otherwise("value_range_fallback"),
    )
    .withColumn(
        "aggregation_method",
        f.when(
            (f.col("kpi_character") == "mean_like") | (f.col("kpi_character") == "ratio"), "avg"
        ).otherwise("sum"),
    )
    .drop("keyword_match")
)

In [ ]:
# ---------------------------------------------------------------
# REVIEW
# ---------------------------------------------------------------

print("=== How many were caught by keywords vs fallback? ===")
kpi_classified.groupBy("classification_source", "kpi_character").count().show()

print("=== KPI Classification Results ===")
kpi_classified.orderBy("kpi_character", "kpi_id").show(400, truncate=False)

In [ ]:
kpi_classified.where(f.col("kpi_character") == "unknown").show(43, truncate=False)

### Single KPI Analysis

### Autocorrelation analysis

In [ ]:
# Define the window: order time within each cell (distname) for lagging
w = Window.partitionBy("bts_id", "distname", "kpi_id").orderBy("start_time")

MAX_LAG = 24

# Filter for KPIs which had been measured more frequent than 24h
df_lagged = regularized_df.where(f.col("natural_freq_minutes") <= 1440)

# Create all lag columns in a loop
for i in range(1, MAX_LAG + 1):
    df_lagged = df_lagged.withColumn(f"lag_{i}", f.lag("kpi_value", i).over(w))

# Calculate correlations
agg_exprs = [f.corr("kpi_value", f"lag_{i}").alias(f"lag_{i}") for i in range(1, MAX_LAG + 1)]

# IMPORTANT Grouping by: kpi_id
acf_spark_df = df_lagged.groupBy("kpi_id").agg(*agg_exprs)

if os.path.exists(f"/home/{user}/app/apps/apps/generator/notebooks/eda/data/acf_spark_df"):
    acf_spark_df = spark.read.parquet(
        f"/home/{user}/app/apps/apps/generator/notebooks/eda/data/acf_spark_df"
    )
else:
    acf_spark_df.write.parquet(
        f"/home/{user}/app/apps/apps/generator/notebooks/eda/data/acf_spark_df", mode="overwrite"
    )


pdf = acf_spark_df.toPandas()

# unpivot
acf_pdf = pdf.melt(id_vars=["kpi_id"], var_name="Lag_Name", value_name="Correlation")

# Clean the Lag column to be just integers
acf_pdf["Lag"] = acf_pdf["Lag_Name"].str.replace("lag_", "").astype(int)

# Create Lag 0 for every KPI
lag_0_df = pd.DataFrame(
    {"kpi_id": pdf["kpi_id"].unique(), "Lag_Name": "lag_0", "Correlation": 1.0, "Lag": 0}
)

final_acf_pdf = pd.concat([lag_0_df, acf_pdf]).sort_values(["kpi_id", "Lag"]).reset_index(drop=True)

# Get one natural frequency per KPI
freq_pdf = (
    regularized_df.select("kpi_id", "natural_freq_minutes")
    .dropna()
    .dropDuplicates(["kpi_id", "natural_freq_minutes"])
    .toPandas()
)

# If one KPI truly has one frequency, this is enough
freq_map = (
    freq_pdf.sort_values(["kpi_id", "natural_freq_minutes"])
    .drop_duplicates(subset=["kpi_id"], keep="first")
    .set_index("kpi_id")["natural_freq_minutes"]
    .to_dict()
)

In [ ]:
u.plot_acf(final_acf_pdf, freq_map)

### Selected KPIs autocorrelation analysis

In [ ]:
kpi_classes = [row[0] for row in kpi_defs_df.select("kpi_class").distinct().collect()]
print(f"Classes: {kpi_classes}")
print(100 * "-")

kpi_cats = [row[0] for row in kpi_defs_df.select("kpi_category").distinct().collect()]
print(f"Categories: {kpi_cats}")
print(100 * "-")

kpi_units = [row[0] for row in kpi_defs_df.select("unit").distinct().collect()]
print(f"Units: {kpi_units}")
print(100 * "-")


kpi_sub_cats = [row[0] for row in kpi_defs_df.select("kpi_sub_category").distinct().collect()]
print(f"Subcategories: {kpi_sub_cats}")
print(100 * "-")

kpi_names = [row[0] for row in kpi_defs_df.select("kpi_name").distinct().collect()]
print(f"Names: {kpi_names}")
print(100 * "-")

In [ ]:
# 1) Define KPI concepts as required tokens
kpi_rules = {
    "prb_util_dl": ["prb", "utilization", "dl"],
    "prb_util_ul": ["prb", "utilization", "ul"],
    "dl_throughput": ["dl", "throughput"],
    "ul_throughput": ["ul", "throughput"],
    "traffic_volume": ["traffic", "volume"],
    "rrc_setup_sr": ["rrc", "setup", "success"],
    "erab_setup_sr": ["erab", "setup", "success"],
    "erab_drop": ["erab", "drop"],
    "handover_sr": ["handover", "success"],
    "cell_availability": ["cell", "availability"],
}


# 2) Normalize KPI names:
def normalize_text(col):
    return f.trim(f.regexp_replace(f.lower(col), r"[^a-z0-9]+", " "))


df = kpi_defs_df.withColumn("kpi_name_norm", normalize_text(f.col("kpi_name")))


# 3) Helper: build an AND condition across tokens
def all_tokens_present(col_name, tokens):
    conditions = [
        f.col(col_name).rlike(rf"(^| ){re.escape(token.lower())}( |$)") for token in tokens
    ]
    return reduce(lambda a, b: a & b, conditions)


# 4) Assign canonical KPI group
expr = None
for group_name, tokens in kpi_rules.items():
    condition = all_tokens_present("kpi_name_norm", tokens)
    expr = (
        f.when(condition, f.lit(group_name))
        if expr is None
        else expr.when(condition, f.lit(group_name))
    )

df_tagged = df.withColumn("kpi_group", expr.otherwise(f.lit(None)))

# 5) Keep only matched KPIs
selected_kpis_df = df_tagged.filter(f.col("kpi_group").isNotNull())

selected_kpis_df.select("kpi_name", "kpi_group").show(truncate=False)

In [ ]:
pm_selected_kpis = regularized_df.join(selected_kpis_df, on="kpi_id", how="inner")

filter KPIs that have NULLs (normally we could try impute them but be careful methods like interpolation could introduce leakage- in case of autocorrelation analysis)

In [ ]:
# Define the window: order time within each cell (distname) for lagging
w = Window.partitionBy("bts_id", "distname", "kpi_id").orderBy("start_time")

MAX_LAG = 24

# Filter for KPIs which had been measured more frequent than 24h
df_lagged = pm_selected_kpis.where(f.col("natural_freq_minutes") <= 1440)

# Create all lag columns in a loop
for i in range(1, MAX_LAG + 1):
    df_lagged = df_lagged.withColumn(f"lag_{i}", f.lag("kpi_value", i).over(w))

# Calculate correlations
agg_exprs = [f.corr("kpi_value", f"lag_{i}").alias(f"lag_{i}") for i in range(1, MAX_LAG + 1)]

# IMPORTANT Grouping by: kpi_id
acf_spark_df = df_lagged.groupBy("kpi_id").agg(*agg_exprs)

pdf = acf_spark_df.toPandas()

# unpivot
acf_pdf = pdf.melt(id_vars=["kpi_id"], var_name="Lag_Name", value_name="Correlation")

# Clean the Lag column to be just integers
acf_pdf["Lag"] = acf_pdf["Lag_Name"].str.replace("lag_", "").astype(int)

# Create Lag 0 for every KPI
lag_0_df = pd.DataFrame(
    {"kpi_id": pdf["kpi_id"].unique(), "Lag_Name": "lag_0", "Correlation": 1.0, "Lag": 0}
)

final_acf_pdf = pd.concat([lag_0_df, acf_pdf]).sort_values(["kpi_id", "Lag"]).reset_index(drop=True)

# Get one natural frequency per KPI
freq_pdf = (
    pm_selected_kpis.select("kpi_id", "natural_freq_minutes")
    .dropna()
    .dropDuplicates(["kpi_id", "natural_freq_minutes"])
    .toPandas()
)

# If one KPI truly has one frequency, this is enough
freq_map = (
    freq_pdf.sort_values(["kpi_id", "natural_freq_minutes"])
    .drop_duplicates(subset=["kpi_id"], keep="first")
    .set_index("kpi_id")["natural_freq_minutes"]
    .to_dict()
)

In [ ]:
# join kpi_groups with corr df
kpi_groups = selected_kpis_df.select("kpi_id", "kpi_group").toPandas()
final_acf_pdf2 = final_acf_pdf.merge(kpi_groups, on="kpi_id", how="left")

print(f"We can choose from kpi_groups of: {final_acf_pdf2['kpi_group'].unique().tolist()}")

In [ ]:
dl_throughput_df = final_acf_pdf2[final_acf_pdf2["kpi_group"] == "dl_throughput"]
ul_throughput_df = final_acf_pdf2[final_acf_pdf2["kpi_group"] == "ul_throughput"]
rrc_setup_sr_df = final_acf_pdf2[final_acf_pdf2["kpi_group"] == "rrc_setup_sr"]
handover_sr = final_acf_pdf2[final_acf_pdf2["kpi_group"] == "handover_sr"]
cell_availability = final_acf_pdf2[final_acf_pdf2["kpi_group"] == "cell_availability"]

In [ ]:
u.plot_acf(dl_throughput_df, freq_map)
u.plot_acf(ul_throughput_df, freq_map)
u.plot_acf(rrc_setup_sr_df, freq_map)
u.plot_acf(handover_sr, freq_map)
u.plot_acf(cell_availability, freq_map)

### KPI Multicollinearity

In [ ]:
from random import sample

# KPI SELECTION
kpi_list = pm_df.select("kpi_id").distinct().rdd.flatMap(lambda x: x).collect()

# PICK HOW MANY KPIS SHOULD BE ANALYZED
KPI_NUMBER = 4

selected_kpi_ids_list = sample(kpi_list, KPI_NUMBER)

group_cols = ["kpi_id"]
MAX_LAG = 24

In [ ]:
kpi_defs_pdf = kpi_defs_df.toPandas()

for skpi in selected_kpi_ids_list:
    for kpi in skpi:
        print(f"{kpi} id: {kpi_defs_pdf[kpi_defs_pdf['kpi_id'] == kpi]['kpi_name'].iloc[0]}")
    corr_df, figs = u.analyze_multicollinearity(regularized_df, group_cols, skpi, MAX_LAG)

SIMPLE REPORTS Analysis

In [ ]:
# count is low, in pandas
simple_reports_pdf = simple_reports_df.toPandas()

In [ ]:
simple_reports_pdf.info()
simple_reports_pdf

In [ ]:
# Report per cell
simple_reports_grby = (
    simple_reports_pdf.groupby(["bts_id", "distname", "report_name"]).size().reset_index()
)
simple_reports_grby

In [ ]:
simple_reports_pdf[
    (simple_reports_pdf["distname"] == simple_reports_grby.iloc[0]["distname"])
    & (simple_reports_pdf["report_name"] == simple_reports_grby.iloc[0]["report_name"])
].sort_values("datetime")

simple reports are CM data per cell changing in time  
for now, there are a total of 5 reports

It is possible to just groupby cell-report_id

In [ ]:
simple_reports_df_g = simple_reports_pdf.groupby(["distname", "report_name"]).agg(
    report_resutotal=("report_result", "size"),
    unique=("report_result", "nunique"),
    duplicates=("report_result", lambda x: x.size - x.nunique()),
)
simple_reports_df_g

Simple reports (cell metadata) to kpi_value statistic correlation

In [ ]:
simple_reports_pivot = simple_reports_df.groupBy("datetime", "bts_id", "distname").pivot("report_name").agg(f.first("report_result"))

In [ ]:
def analyze_kpi_vs_metadata(df_kpi, df_meta):
    pass

In [ ]:
analyze_kpi_vs_metadata(pm_df, simple_reports_df)

obsluga wersji



In [ ]:
u.plot_kpi_versions_selector(pm_df, group_col="bts_id") #, group_col="distname" per distname 

Wersje w kpi_definitions

In [ ]:
kpi_defs_df = kpi_defs_df.withColumn("base_kpi", f.regexp_extract("kpi_id", r"^(.*?)[a-z]?$", 1))

In [ ]:
kpi_defs_df.printSchema()

In [ ]:
kpi_ver_group = kpi_defs_df.groupby("base_kpi").agg(
    *[
        f.countDistinct(col).alias(col) for col in ["kpi_name", "technology", "unit", "kpi_category", "kpi_sub_category", "kpi_class"]
    ]
)

In [ ]:
for col in ["kpi_name", "technology", "unit", "kpi_category", "kpi_sub_category", "kpi_class"]:
    print(f"{col}: {kpi_ver_group.filter(f.col(col) > 1).count()}")
    kpi_ver_group.filter(f.col(col) > 1).show()

In [ ]:
KPI_NAME = "<kpi_name>"
kpi_defs_df.filter(f.col("base_kpi") == KPI_NAME).show(truncate=False)

In [ ]:
# PM kpi on simple reports

In [ ]:
window_spec = Window.partitionBy("bts_id", "distname", "report_name").orderBy(f.col("datetime").desc())

latest_reports_df = simple_reports_df.withColumn("rn", f.row_number().over(window_spec)) \
    .filter(f.col("rn") == 1) \
    .drop("rn", "datetime", "report_id")

metadata_pivoted = latest_reports_df.groupBy("bts_id", "distname").pivot("report_name").agg(
    f.first("report_result")
)

pm_kpi_pivot_on_simple_reports_df = pm_kpi_pivot_df.join(
    f.broadcast(metadata_pivoted),
    on=["bts_id", "distname"],
    how="inner" 
)


In [ ]:
kpi_columns = sorted([kpi_id for kpi_id in pm_kpi_pivot_on_simple_reports_df.columns if "NR_" in kpi_id]) # regex invented in 2000, ppl in 1999
metadata_columns = sorted([col for col in pm_kpi_pivot_on_simple_reports_df.columns if 'CELL' in col])
cols = kpi_columns + metadata_columns

kpi_reports_df = pm_kpi_pivot_on_simple_reports_df.select(*cols) \
    .sample(withReplacement=False, fraction=0.2, seed=42) \
    .toPandas()

In [ ]:
u.visualize_kpi_on_simple_reports_groups(kpi_reports_df, kpi_columns, metadata_columns)

In [ ]:
u.visualize_heatmap_kpi_on_simple_reports_groups(kpi_reports_df, kpi_columns, metadata_columns)